In [64]:
"""
ST-GCN. Read https://thachngoctran.medium.com/spatial-temporal-graph-convolutional-networks-st-gcn-explained-bf926c811330
When adding another dimension to the nodes, such as time, it is motivating to convolve in both spatial and temporal dimensions
"""

"""
Also test the following:
STGCN — graph conv + 1D temporal conv. Simplest, fastest, great baseline. Start here.
DCRNN — diffusion conv + GRU; very well-studied on traffic networks.
Graph WaveNet — learns the adjacency instead of using a fixed one; strong when your hand-built graph is suspect.
"""

'\nAlso test the following:\nSTGCN — graph conv + 1D temporal conv. Simplest, fastest, great baseline. Start here.\nDCRNN — diffusion conv + GRU; very well-studied on traffic networks.\nGraph WaveNet — learns the adjacency instead of using a fixed one; strong when your hand-built graph is suspect.\n'

In [65]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [66]:
df = pd.read_parquet("data/interim/rides_wide.parquet")

In [67]:
df.shape

(2267, 7321)

In [68]:
df.head()

,2025-05-30 23:00:00,2025-05-31 00:00:00,2025-05-31 01:00:00,2025-05-31 02:00:00,2025-05-31 03:00:00,2025-05-31 04:00:00,2025-05-31 05:00:00,2025-05-31 06:00:00,2025-05-31 07:00:00,2025-05-31 08:00:00,...,2026-03-31 14:00:00,2026-03-31 15:00:00,2026-03-31 16:00:00,2026-03-31 17:00:00,2026-03-31 18:00:00,2026-03-31 19:00:00,2026-03-31 20:00:00,2026-03-31 21:00:00,2026-03-31 22:00:00,2026-03-31 23:00:00
start_station_id,,,,,,,,,,,,,,,,,,,,,
1234.56,0,0,0,0,0,0,0,0,0,0,...,1,1,1,0,0,0,0,0,3,0
1964.01,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2009.04,0,0,0,0,0,0,0,0,0,0,...,1,1,4,0,4,0,0,0,0,0
2042.01,0,0,0,0,0,0,0,0,0,0,...,0,0,2,0,0,0,0,0,0,0
2086.07,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [69]:
X = df.copy()
X = X.T

In [70]:
n_hours = X.shape[0]

In [71]:
train_end = int(n_hours* 0.7)
val_end = int(n_hours * 0.85)

In [72]:
X_train = X.iloc[:train_end] # all rows, all cols until train end
X_val   = X.iloc[train_end:val_end] # all rows, all cols between train end and val end
X_test  = X.iloc[val_end:] # all rows, all cols after val end


In [73]:
# BASELINE 1
# Per-station mean demand by (dow, hour), fit on train only
dow_tr  = X_train.index.dayofweek # 1-6
hour_tr = X_train.index.hour # 0-23
baseline = X_train.groupby([dow_tr, hour_tr]).mean()      # (dow, hour) × stations for train data

# predict for val
dow_val  = X_val.index.dayofweek
hour_val = X_val.index.hour
y_hat_val = baseline.loc[list(zip(dow_val, hour_val))]      # zipping baseline --> rows: val timestamps, cols: stations
y_hat_val.index = X_val.index
   

In [74]:
err = (X_val - y_hat_val).abs() # baseline contains the meean, while the real data may be different

mae_overall   = err.values.mean()                    # one scalar across all cells
mae_per_stn   = err.mean(axis=0)                     # Series, len 2236 — one MAE per station
mae_per_hour  = err.mean(axis=1)                     # Series, len 216  — one MAE per val timestamp

print(f"Overall MAE: {mae_overall:.4f}")
print(f"Median per-station MAE: {mae_per_stn.median():.4f}")


Overall MAE: 1.4936
Median per-station MAE: 0.8277
